# Project 3: Synthetic Data Studio - AI-Powered Tabular Dataset Generator

## Executive Summary

This project showcases a **production-grade synthetic data generation system** powered by Large Language Models. It bridges the critical gap between data needs and data availability by creating realistic, structured datasets from natural language descriptions.

**Core Innovation:**
The system leverages state-of-the-art open-source LLMs (Llama 3.1 8B, Llama 3.2 3B, Gemma 2) to generate **tabular CSV datasets** that adhere to strict business rules and data schemas. Users simply describe their domain (e.g., retail sales, banking, customer support) and the AI produces fully-formed, downloadable datasets.

**Key Capabilities:**
1.  **Schema-Driven Generation**: Pre-defined business templates with column specifications, data types, and constraints.
2.  **Model Quantization**: 4-bit quantization using BitsAndBytes for efficient local deployment on consumer GPUs.
3.  **Quality Validation**: Automated checks for row count, column presence, and data type conformance.
4.  **Interactive Web UI**: Gradio interface allowing non-technical users to generate data without writing code.

**Business Value:**
- **Data Privacy**: Generate de-identified datasets for demos and prototypes without exposing real customer data.
- **ML Development**: Create augmented training data for machine learning pipelines when real data is scarce.
- **Testing & QA**: Populate testing environments with realistic data at any scale.

**Technical Stack:**
- **LLM**: Llama 3.1 8B Instruct (4-bit quantized) for high-quality text-to-data generation.
- **Inference**: Hugging Face Transformers with BitsAndBytes quantization.
- **UI**: Gradio for rapid prototyping and deployment.
- **Data Processing**: Pandas for DataFrame manipulation and CSV export.

## 1. Environment Setup & Dependencies Installation

We begin by installing the complete technology stack required for this project:

- **transformers**: Hugging Face library for loading and running LLMs.
- **huggingface_hub**: Authentication and model downloading.
- **bitsandbytes**: Enables 4-bit quantization for memory-efficient inference.
- **torch**: PyTorch framework for deep learning operations.
- **gradio**: Rapid UI prototyping for ML applications.
- **pandas**: Data manipulation and CSV generation.
- **dotenv**: Secure environment variable management.
- **openai**: Optional client for cloud-based models (if needed).

In [ ]:
# %pip install -q transformers huggingface_hub gradio pandas bitsandbytes python-dotenv torch

In [3]:
%run ../config/llm_settings.py

In [4]:
%run ../utils/llm_functions.py

In [5]:
import re
import gradio as gr
from datetime import datetime
from pathlib import Path

import torch
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from IPython.display import Markdown, display, update_display

## 2. Authentication & Environment Configuration

**Security Best Practices:**
We load API keys from a `.env` file to avoid hardcoding credentials in the codebase.

**Dual Authentication:**
- **OpenAI Client** (optional): For hybrid scenarios where cloud models supplement local inference.
- **Hugging Face Login**: Required to download gated models like Llama from Meta's official repository.

**Bearer Token Handling:**
The code automatically strips the `Bearer` prefix if present, preventing authentication errors.

In [6]:
hf_token = HF_TOKEN

if hf_token and hf_token.startswith("Bearer "):
    hf_token = hf_token.replace("Bearer ", "", 1).strip()

if not hf_token:
    raise RuntimeError("HF_TOKEN no está configurado.")

print("HF_TOKEN loaded:", bool(hf_token))

# login(hf_token, add_to_git_credential=True)

HF_TOKEN loaded: True


## 3. Model Inventory & Selection

This project supports multiple state-of-the-art instruction-tuned models:

**Available Models:**
- **Llama 3.1 8B Instruct**: Meta's flagship model with excellent instruction following (8 billion parameters).
- **Llama 3.2 3B Instruct**: Smaller, faster variant suitable for resource-constrained environments.
- **Phi-3 Mini 4K**: Microsoft's compact model optimized for 4K context windows.
- **Gemma 2 9B IT**: Google's instruction-tuned model with strong reasoning capabilities.

**Active Selection:**
We use **Llama 3.1 8B** in 4-bit quantization as the primary model due to its optimal balance between quality and resource efficiency.

In [7]:
# Modelos locales y parámetros del proyecto, configurables desde apis.env.
LLAMA_3_1 = env(
    "PROJECT3_MODEL_LLAMA_3_1",
    "meta-llama/Llama-3.1-8B-Instruct",
)
LLAMA_3_2 = env(
    "PROJECT3_MODEL_LLAMA_3_2",
    "meta-llama/Llama-3.2-3B-Instruct",
)
PHI4 = env(
    "PROJECT3_MODEL_PHI4",
    "microsoft/Phi-3-mini-4k-instruct",
)
GEMMA3 = env(
    "PROJECT3_MODEL_GEMMA3",
    "google/gemma-3-1b-it",
)

PROJECT3_MAX_NEW_TOKENS = int(env("PROJECT3_MAX_NEW_TOKENS", "4096"))
PROJECT3_TEMPERATURE = float(env("PROJECT3_TEMPERATURE", "0.7"))
PROJECT3_TOP_P = float(env("PROJECT3_TOP_P", "0.9"))

## 4. Cache Management Strategy

**Problem:** Large language models (8GB+) can fill up default cache partitions, causing storage issues.

**Solution:** Dedicated cache directories for each model.

**Benefits:**
1. **Isolation**: Each model's weights are stored separately, preventing cross-contamination.
2. **Disk Management**: Easier to identify and delete specific models when storage is needed.
3. **Multi-Model Workflows**: Safely swap between models without re-downloading.

**Implementation:**
We define a base cache directory from the `HF_HOME` environment variable and create subdirectories for each model family.

In [8]:
# Rutas del proyecto y cache de Hugging Face.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "outputs").is_dir() and (PROJECT_ROOT.parent / "outputs").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUTS_DIR = PROJECT_ROOT / "inputs"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

hf_cache_base = Path(
    env("HF_HOME", str(Path.home() / ".cache" / "huggingface"))
)
model_cache_base = hf_cache_base / "models"
model_cache_llama_3_1 = model_cache_base / "llama_3_1_8b"
model_cache_llama_3_2 = model_cache_base / "Llama-3.2-3B"
model_cache_phi3 = model_cache_base / "phi_3_mini"
model_cache_gemma_3 = model_cache_base / "gemma_3_4b"

for cache_dir in (
    model_cache_llama_3_1,
    model_cache_phi3,
    model_cache_gemma_3,
    model_cache_llama_3_2,
):
    cache_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Inputs: {INPUTS_DIR}")
print(f"Outputs: {OUTPUTS_DIR}")
print(f"Llama cache: {model_cache_llama_3_1}")
print(f"Llama 3.2 cache: {model_cache_llama_3_2}")
print(f"Phi-3 cache: {model_cache_phi3}")
print(f"Gemma cache: {model_cache_gemma_3}")

Project root: /home/jovyan/work/llms-engineering-main
Inputs: /home/jovyan/work/llms-engineering-main/inputs
Outputs: /home/jovyan/work/llms-engineering-main/outputs
Llama cache: /home/jovyan/.cache/huggingface/models/llama_3_1_8b
Llama 3.2 cache: /home/jovyan/.cache/huggingface/models/Llama-3.2-3B
Phi-3 cache: /home/jovyan/.cache/huggingface/models/phi_3_mini
Gemma cache: /home/jovyan/.cache/huggingface/models/gemma_3_4b


## 5. Business Domain Templates (Schema Definitions)

This is the **knowledge base** of the system. Each template defines a real-world business domain with:
- **Description**: Context about the dataset's purpose.
- **Columns**: List of fields with name, data type, and validation constraints.

**Template 1: Retail Sales**
E-commerce transaction data with fraud detection capabilities.

**Template 2: Bank Transactions**
Financial movements with balance tracking and multi-channel support.

**Template 3: Customer Support Tickets**
SaaS help desk data with priority levels and resolution metrics.

**Why This Matters:**
These schemas act as **structured prompts** for the LLM, ensuring generated data is not only realistic but also business-compliant. Constraints like "unique", "date range", and "category" guide the model to produce valid outputs.

In [9]:
DATASET_SCHEMAS = {
    "Retail Sales": {
        "description": "E-commerce retail sales transactions with fraud detection.",
        "columns": [
            {"name": "order_id", "type": "string", "constraints": "unique, format ORD-XXXX"},
            {"name": "order_date", "type": "date", "constraints": "between 2024-01-01 and 2024-12-31"},
            {"name": "customer_id", "type": "string", "constraints": "format CUST-XXXX"},
            {"name": "country", "type": "category", "constraints": "Colombia, Mexico, Chile, Peru"},
            {"name": "product_category", "type": "category", "constraints": "Electronics, Clothing, Home"},
            {"name": "unit_price", "type": "float", "constraints": "between 5 and 200"},
            {"name": "quantity", "type": "int", "constraints": "between 1 and 10"},
            {"name": "total_amount", "type": "float", "constraints": "unit_price * quantity"},
            {"name": "is_fraud", "type": "bool", "constraints": "True if transaction is fraudulent, False otherwise"}
        ]
    },
    "Bank Transactions": {
        "description": "Banking transactions for savings accounts.",
        "columns": [
            {"name": "transaction_id", "type": "string", "constraints": "unique"},
            {"name": "customer_id", "type": "string", "constraints": "format CUST-XXXX"},
            {"name": "transaction_date", "type": "date", "constraints": "2024-01-01 to 2024-12-31"},
            {"name": "transaction_type", "type": "category", "constraints": "deposit, withdrawal, transfer"},
            {"name": "amount", "type": "float", "constraints": "between 10 and 5000"},
            {"name": "balance_after", "type": "float", "constraints": "coherent balance after transaction"},
            {"name": "channel", "type": "category", "constraints": "ATM, web, mobile_app, branch"}
        ]
    },
    "Customer Support Tickets": {
        "description": "Support tickets for a SaaS platform.",
        "columns": [
            {"name": "ticket_id", "type": "string", "constraints": "unique"},
            {"name": "created_at", "type": "datetime", "constraints": "2024-01-01 to 2024-12-31"},
            {"name": "customer_tier", "type": "category", "constraints": "Free, Standard, Premium"},
            {"name": "issue_type", "type": "category", "constraints": "bug, billing, onboarding, other"},
            {"name": "priority", "type": "category", "constraints": "low, medium, high, critical"},
            {"name": "resolution_time_hours", "type": "float", "constraints": ">= 0"},
            {"name": "resolved", "type": "bool", "constraints": "True/False"}
        ]
    }
}

## 6. Prompt Engineering for Data Generation

**The Challenge:**
LLMs are conversational by nature. Getting them to produce *only* raw CSV data (no explanations, no markdown) requires precise prompt construction.

**Prompt Components:**
1. **Role Definition**: "You are a synthetic data generator..."
2. **Task Specification**: Dataset name, description, and purpose.
3. **Schema Injection**: Programmatically insert column specifications from the template.
4. **Output Constraints**:
   - Exact row count requirement
   - CSV-only output (no text before/after)
   - Mandatory header row
   - Strict data type adherence

**Extra Instructions:**
Users can add custom rules like "Generate 10% fraudulent transactions" to fine-tune the output.

This function builds the complete prompt dynamically based on the selected schema and user inputs.

In [10]:
def build_prompt(schema_name: str, n_rows: int, extra_instructions: str = "") -> str:
    """
    Constructs a detailed prompt for the LLM to generate synthetic tabular data.
    
    Args:
        schema_name: Name of the business domain template.
        n_rows: Number of rows to generate.
        extra_instructions: Optional user-defined constraints.
    
    Returns:
        Complete prompt string ready for LLM inference.
    """
    schema = DATASET_SCHEMAS[schema_name]
    lines = []

    # Define the AI's role and task
    lines.append(
        "You are a synthetic tabular data generator for analytics and machine learning testing."
    )
    lines.append(
        "Your task is to generate a SYNTHETIC dataset in CSV format, without real personal data."
    )
    lines.append(f"Dataset: {schema_name}")
    lines.append(f"Description: {schema['description']}")
    lines.append("")
    lines.append("Column Specifications:")

    # Inject schema details
    for col in schema["columns"]:
        lines.append(
            f"- {col['name']} ({col['type']}): {col['constraints']}"
        )

    lines.append("")
    lines.append(f"Generate exactly {n_rows} rows of data. You MUST produce {n_rows} rows.")
    lines.append("Do not write any text before or after the CSV. Only the CSV.")
    lines.append("Very important:")
    lines.append("1. Output MUST be in CSV format only.")
    lines.append("2. First row must be the header with column names.")
    lines.append("3. Do not include explanations, comments, or additional text.")
    lines.append("4. Respect data types and ranges as best as possible.")
    
    if extra_instructions:
        lines.append("")
        lines.append("Additional user instructions:")
        lines.append(extra_instructions)

    return "\n".join(lines)

## 7. Tokenizer Configuration

**What is a Tokenizer?**
The tokenizer converts human-readable text into numerical tokens that the LLM can process.

**Critical Configurations:**
1. **Padding Token**: Set to `eos_token` to handle variable-length inputs.
2. **Padding Side**: `"left"` is recommended for causal (decoder-only) models like Llama.

**Why This Matters:**
Incorrect tokenizer settings can cause:
- Silent errors where the model ignores parts of the prompt.
- Shape mismatches in tensor operations.
- Degraded generation quality.

**Checkpoint:**
This cell **must** be executed before calling `generate_with_local_llama`, otherwise you'll encounter `NameError: tokenizer_llama is not defined`.

In [11]:
tokenizer_llama = AutoTokenizer.from_pretrained(
    LLAMA_3_2,
    cache_dir=model_cache_llama_3_2,
)

tokenizer_llama.pad_token = tokenizer_llama.eos_token
tokenizer_llama.padding_side = "left"

print("Tokenizer loaded successfully")

Tokenizer loaded successfully


## 8. Model Quantization with BitsAndBytes

**The Problem:**
Running an 8-billion parameter model in full precision (FP32) requires ~32GB of VRAM, putting it out of reach for most consumer GPUs.

**The Solution: 4-bit Quantization**
Using BitsAndBytes, we compress the model to use only ~4GB of VRAM with minimal quality loss.

**Configuration:**
- **load_in_4bit**: Activates 4-bit precision.
- **bnb_4bit_use_double_quant**: Double quantization for further compression.
- **bnb_4bit_compute_dtype**: `bfloat16` for stable numerical computation.
- **bnb_4bit_quant_type**: `nf4` (NormalFloat4) - optimized for neural networks.

**Impact:**
This allows running enterprise-grade models on a single RTX 3090 or even a laptop with a decent GPU.

In [12]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

## 9. Loading the Llama 3.1 8B Model

**Model Loading Strategy:**
- **device_map="auto"**: Automatically distributes model layers across available devices (GPU, CPU, disk).
- **quantization_config**: Applies the 4-bit compression we configured.
- **cache_dir**: Uses our dedicated storage path.

**What Happens Under the Hood:**
1. Checks if model exists locally in the cache directory.
2. If not, downloads ~4.5GB of quantized weights from Hugging Face.
3. Loads layers into GPU memory (or splits across GPU/CPU if needed).
4. Returns a ready-to-use model object.

**Execution Time:**
- First run (download): 5-15 minutes depending on internet speed.
- Subsequent runs (cached): 30-60 seconds.

In [13]:
MODEL_LLAMA = AutoModelForCausalLM.from_pretrained(
    LLAMA_3_2,
    device_map="auto",
    quantization_config=quant_config,
    cache_dir=model_cache_llama_3_2,
)

print(f"Model loaded successfully from: {model_cache_llama_3_2}")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model loaded successfully from: /home/jovyan/.cache/huggingface/models/Llama-3.2-3B


## 10. Inference Function with Memory Management

**Core Generation Logic:**
This function is the bridge between our prompt and the model's output.

**Key Parameters:**
- **temperature (0.7)**: Controls randomness. Lower = more deterministic, higher = more creative.
- **top_p (0.9)**: Nucleus sampling - considers only the top 90% probability tokens.
- **do_sample (True)**: Enables sampling (vs greedy decoding) for diverse outputs.
- **max_new_tokens (4096)**: Maximum length of generated response.

**Critical Memory Management:**
After generation, we explicitly:
1. Delete intermediate tensors (`inputs`, `generated_ids`).
2. Call `torch.cuda.empty_cache()` to release GPU memory.

**Why This Matters:**
Without cleanup, GPU memory can fragment, causing "CUDA out of memory" errors on subsequent runs or larger datasets.

**Output Extraction:**
We separate the generated text from the original prompt using string slicing, returning only the model's response.

In [14]:
def generate_with_local_llama(
    prompt: str,
    max_new_tokens: int | None = None,
    temperature: float | None = None,
    top_p: float | None = None,
) -> str:
    """Generate text using the configured local model."""
    if not prompt or not prompt.strip():
        raise ValueError("prompt must be a non-empty string.")

    resolved_max_new_tokens = (
        PROJECT3_MAX_NEW_TOKENS
        if max_new_tokens is None
        else int(max_new_tokens)
    )
    resolved_temperature = (
        PROJECT3_TEMPERATURE if temperature is None else float(temperature)
    )
    resolved_top_p = PROJECT3_TOP_P if top_p is None else float(top_p)

    if resolved_max_new_tokens < 1:
        raise ValueError("max_new_tokens must be greater than zero.")
    if not 0.0 < resolved_top_p <= 1.0:
        raise ValueError("top_p must be greater than 0 and at most 1.")
    if resolved_temperature < 0:
        raise ValueError("temperature must be non-negative.")

    inputs = tokenizer_llama(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(MODEL_LLAMA.device)

    with torch.no_grad():
        generated_ids = MODEL_LLAMA.generate(
            **inputs,
            max_new_tokens=resolved_max_new_tokens,
            temperature=resolved_temperature,
            top_p=resolved_top_p,
            do_sample=resolved_temperature > 0,
            pad_token_id=tokenizer_llama.eos_token_id,
        )

    full_text = tokenizer_llama.decode(
        generated_ids[0],
        skip_special_tokens=True,
    )

    del inputs, generated_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    generated_part = full_text[len(prompt):].strip()
    return generated_part if generated_part else full_text

## 11. CSV Parsing & Data Cleaning

**The Challenge:**
LLMs don't always return perfectly formatted CSV. They might wrap it in markdown code blocks (` ```csv ... ``` `), add explanatory text, or include extra newlines.

**Robust Parsing Strategy:**
1. **Strip Markdown**: Remove ` ```csv ` and ` ``` ` markers using regex.
2. **Filter Lines**: Keep only lines containing commas (likely CSV rows).
3. **Pandas Conversion**: Use `pd.read_csv()` with `StringIO` to parse in-memory.

**Error Handling:**
If parsing fails, we:
- Print the error and the problematic content (first 500 chars for debugging).
- Return an empty DataFrame rather than crashing.

**Result:**
A clean pandas DataFrame ready for analysis or export.

In [15]:
import io

def parse_csv_to_df(text: str) -> pd.DataFrame:
    """
    Parses raw LLM output into a pandas DataFrame.
    
    Handles common LLM quirks:
    - Markdown code blocks (```csv ... ```)
    - Extra explanatory text
    - Inconsistent formatting
    
    Args:
        text: Raw string output from the LLM.
    
    Returns:
        pandas DataFrame if parsing succeeds, empty DataFrame otherwise.
    """
    # Remove markdown code fence markers
    cleaned = re.sub(r"```(?:csv)?", "", text)
    cleaned = cleaned.strip("` \n")

    # Filter lines that look like CSV (contain commas)
    lines = [l for l in cleaned.splitlines() if "," in l]
    if not lines:
        print("WARNING: No comma-separated lines found in model output.")
        return pd.DataFrame()

    csv_text = "\n".join(lines)

    try:
        df = pd.read_csv(io.StringIO(csv_text))
    except Exception as e:
        print(f" Error parsing CSV: {e}")
        print("Content attempted to parse:")
        print(csv_text[:500])
        return pd.DataFrame()

    return df

## 12. Data Quality Validation

**Automated Quality Checks:**

This function performs schema validation to ensure the generated data meets expectations.

**Validation Metrics:**
1. **Missing Columns**: Columns defined in the schema but absent in the DataFrame.
2. **Extra Columns**: Columns in the DataFrame not defined in the schema (possible hallucinations).
3. **Row Count**: Number of rows generated (should match user's request).
4. **Column Count**: Total number of columns present.

**Use Cases:**
- **Immediate Feedback**: Alert users if the model deviated from instructions.
- **Pipeline Integration**: Automated tests for CI/CD workflows.
- **Model Monitoring**: Track generation quality over time to detect model drift.

**Example Output:**
```python
{
    "missing_columns": ["is_fraud"],
    "extra_columns": ["timestamp"],
    "n_rows": 95,
    "n_cols": 9
}
```

In [16]:
def basic_quality_checks(df: pd.DataFrame, schema_name: str) -> dict:
    """
    Validates generated DataFrame against the expected schema.
    
    Args:
        df: Generated pandas DataFrame.
        schema_name: Name of the schema template used.
    
    Returns:
        Dictionary containing validation results:
        - missing_columns: Expected columns not in DataFrame
        - extra_columns: Columns in DataFrame not in schema
        - n_rows: Actual row count
        - n_cols: Actual column count
    """
    schema = DATASET_SCHEMAS[schema_name]
    expected_cols = [c["name"] for c in schema["columns"]]

    result = {
        "missing_columns": [c for c in expected_cols if c not in df.columns],
        "extra_columns": [c for c in df.columns if c not in expected_cols],
        "n_rows": len(df),
        "n_cols": df.shape[1]
    }
    return result

## 13. End-to-End Application Pipeline

**The Orchestrator:**
This function ties together all components into a single, user-facing workflow.

**Pipeline Stages:**

**Stage 1: Prompt Construction**
- Calls `build_prompt()` with schema, row count, and user instructions.

**Stage 2: Model Inference**
- Sends prompt to `generate_with_local_llama()`.
- Streams tokens from the model.

**Stage 3: Parsing**
- Converts raw text to DataFrame via `parse_csv_to_df()`.

**Stage 4: Validation**
- Runs quality checks via `basic_quality_checks()`.

**Stage 5: Export**
- Writes DataFrame to a temporary CSV file for download.
- Returns info summary, DataFrame preview, and file path.

**Debug Mode:**
Includes `print()` statements to display:
- The constructed prompt (first 1000 chars).
- Raw model output (first 1000 chars).
- DataFrame shape and head.

**Production Tip:**
In a real deployment, replace `print()` with proper logging (`logging.info()`) for better observability.

## 14. Interactive Web Interface with Gradio

**User Experience Design:**

**Input Controls:**
1. **Dropdown**: Select from 3 pre-defined business schemas.
2. **Slider**: Choose number of rows (10-1000, step of 10).
3. **Textbox**: Add custom instructions (e.g., "Generate 10% fraudulent transactions").

**Output Components:**
1. **Textbox**: Displays validation summary (rows generated, missing columns, etc.).
2. **Dataframe**: Interactive preview of the first few rows.
3. **File**: Direct CSV download button.

**Event Binding:**
The "Generate" button triggers `synthetic_data_app()`, which:
- Takes all three inputs.
- Runs the full pipeline.
- Updates all three outputs simultaneously.

**Deployment:**
`share=True` creates a temporary public URL (via Gradio's tunneling service), allowing you to:
- Demo the tool to clients without local setup.
- Test from mobile devices.
- Share with non-technical stakeholders.

In [17]:
def synthetic_data_app(
    schema_name: str,
    n_rows: int,
    extra_instructions: str = "",
):
    """Generate, validate, and save a synthetic dataset."""
    n_rows = int(n_rows)
    if n_rows < 1:
        raise ValueError("n_rows must be greater than zero.")
    if schema_name not in DATASET_SCHEMAS:
        raise ValueError(f"Unknown schema: {schema_name}")

    prompt = build_prompt(schema_name, n_rows, extra_instructions or "")
    print("=== PROMPT (first 1000 chars) ===")
    print(prompt[:1000])
    print("=================================")

    raw_output = generate_with_local_llama(prompt)
    print("=== RAW OUTPUT (first 1000 chars) ===")
    print(raw_output[:1000])
    print("=====================================")

    df = parse_csv_to_df(raw_output)
    checks = basic_quality_checks(df, schema_name)

    output_name = (
        f"synthetic_{schema_name.lower().replace(' ', '_')}_"
        f"{datetime.now():%Y%m%d_%H%M%S}.csv"
    )
    output_path = OUTPUTS_DIR / output_name
    df.to_csv(output_path, index=False)

    info = (
        f"Rows generated: {checks['n_rows']}\n"
        f"Extra columns: {checks['extra_columns']}\n"
        f"Missing columns: {checks['missing_columns']}\n"
        f"Saved to: {output_path}"
    )

    print("=== DATAFRAME SHAPE ===", df.shape)
    print(df.head())
    return info, df, str(output_path)

In [18]:
gr.close_all()

with gr.Blocks(title="Synthetic Data Studio") as demo:
    gr.Markdown(
        "# Synthetic Data Studio\n"
        "AI-Powered Tabular Dataset Generator using a local LLM"
    )

    schema_name = gr.Dropdown(
        choices=list(DATASET_SCHEMAS.keys()),
        value="Retail Sales",
        label="Dataset Type",
    )
    n_rows = gr.Slider(
        minimum=10,
        maximum=1000,
        value=100,
        step=10,
        label="Number of Rows",
    )
    extra_instructions = gr.Textbox(
        lines=4,
        label="Additional Instructions (optional)",
        placeholder="E.g., Generate 10% fraudulent transactions...",
    )
    generate_btn = gr.Button("Generate Synthetic Data", variant="primary")

    info_out = gr.Textbox(label="Generation Summary")
    df_out = gr.Dataframe(label="Dataset Preview")
    csv_out = gr.File(label="Download CSV")

    generate_btn.click(
        fn=synthetic_data_app,
        inputs=[schema_name, n_rows, extra_instructions],
        outputs=[info_out, df_out, csv_out],
    )

demo.launch(
    share=True,
    allowed_paths=[str(OUTPUTS_DIR)],
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://e1c11d0dc283ff8736.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


=== PROMPT (first 1000 chars) ===
You are a synthetic tabular data generator for analytics and machine learning testing.
Your task is to generate a SYNTHETIC dataset in CSV format, without real personal data.
Dataset: Retail Sales
Description: E-commerce retail sales transactions with fraud detection.

Column Specifications:
- order_id (string): unique, format ORD-XXXX
- order_date (date): between 2024-01-01 and 2024-12-31
- customer_id (string): format CUST-XXXX
- country (category): Colombia, Mexico, Chile, Peru
- product_category (category): Electronics, Clothing, Home
- unit_price (float): between 5 and 200
- quantity (int): between 1 and 10
- total_amount (float): unit_price * quantity
- is_fraud (bool): True if transaction is fraudulent, False otherwise

Generate exactly 100 rows of data. You MUST produce 100 rows.
Do not write any text before or after the CSV. Only the CSV.
Very important:
1. Output MUST be in CSV format only.
2. First row must be the header with column names.
3

In [19]:
# 🧹 GPU Memory Cleanup

import gc

# Delete model and tokenizer from global namespace
for var_name in ["MODEL_LLAMA", "tokenizer_llama"]:
    try:
        del globals()[var_name]
        print(f"✅ Deleted: {var_name}")
    except KeyError:
        print(f"⚠️ {var_name} not found in globals()")

# Force garbage collection
gc.collect()
gr.close_all()

# Clear CUDA cache if available
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ CUDA cache cleared")
else:
    print("ℹ️ CUDA not available in this environment")

✅ Deleted: MODEL_LLAMA
✅ Deleted: tokenizer_llama
Closing server running on port: 7860
✅ CUDA cache cleared
